# Chân dung khách hàng: phân nhóm từ lịch sử mua
**Sau bài 08 · 3–4 giờ, làm cá nhân hoặc cặp.** Đây là thời gian dự án bổ sung, không cộng vào 150 phút của bài học.

**Sản phẩm:** Tạo RFM từ hóa đơn, phân cụm và xuất bảng khách hàng cùng hồ sơ nhóm có diễn giải.

Đọc → dự đoán → chạy mã mẫu → tự điền TODO → kiểm thử → giải thích. Các TODO cố ý dừng bằng `NotImplementedError`; đây không phải lỗi môi trường. Không thay kiểm thử bằng giá trị viết cứng. Giữ seed, Restart Kernel và chạy lại trước khi nộp.

Nguồn, điều kiện sử dụng và giới hạn dữ liệu: [bảng nguồn](../../course/DATA_SOURCES.md). Không tải mô hình pickle/joblib không rõ nguồn.


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'mlcourse').is_dir()), None)
assert ROOT is not None, 'Mở notebook trong bản tải đầy đủ của repository.'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from threadpoolctl import threadpool_limits
_threads = threadpool_limits(limits=2)
SEED = 42

OUT = ROOT/'outputs/projects/04_customer_segments'
OUT.mkdir(parents=True, exist_ok=True)


## 1. Phân nhóm để làm gì?
Tìm nhóm hành vi mua trong lịch sử để đề xuất cách tìm hiểu khách hàng. Đây là phân tích mô tả, không dự báo nhân quả hay khẳng định khách nào sẽ mua tiếp.
R: số ngày từ giao dịch gần nhất đến 10/12/2011. F: số hóa đơn khác nhau. M: tổng Quantity × UnitPrice của các dòng giữ lại.


In [ ]:
import zipfile
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
import joblib
with zipfile.ZipFile(ROOT/'course/bai08/data/online_retail.zip') as z:
    name=next(n for n in z.namelist() if n.endswith('.xlsx'))
    raw=pd.read_excel(z.open(name))
valid=raw.CustomerID.notna() & raw.Quantity.gt(0) & raw.UnitPrice.gt(0) & ~raw.InvoiceNo.astype(str).str.lower().str.startswith('c')
clean=raw.loc[valid].copy()
clean['CustomerID']=clean.CustomerID.astype('int64')
clean['InvoiceNo']=clean.InvoiceNo.astype(str)
print('Gốc / giữ lại:',len(raw),len(clean))
display(clean.head())


## 2. TODO: gom dòng hàng thành một khách
Trả DataFrame index CustomerID, cột recency, frequency, monetary. F đếm hóa đơn **khác nhau**, không đếm dòng. M không phải lợi nhuận. Giữ dòng trùng theo quy ước học liệu, ghi giới hạn này trong báo cáo.


In [ ]:
def make_rfm(lines, snapshot=pd.Timestamp('2011-12-10')):
    # TODO: line_value; groupby CustomerID; recency dùng ngày lịch (normalize).
    raise NotImplementedError('Hoàn thiện RFM')


Gợi ý: `groupby(...).agg(...)`, `nunique`, `sum`, `max`. Recency chỉ dùng ngày để không mất một ngày do giờ giao dịch.


In [ ]:
tiny=pd.DataFrame({'CustomerID':[1,1,1,2],'InvoiceNo':['A','A','B','C'],'InvoiceDate':pd.to_datetime(['2011-12-01','2011-12-01','2011-12-09','2011-12-08']),'Quantity':[1,2,1,1],'UnitPrice':[10,5,20,7]})
small=make_rfm(tiny)
assert small.loc[1,'frequency']==2 and small.loc[1,'monetary']==40
assert small.loc[1,'recency']==1 and small.loc[2,'recency']==2
rfm=make_rfm(clean)
assert rfm.index.is_unique and (rfm>0).all().all()
display(rfm.describe())


## 3. Mã mẫu: giảm lệch và chuẩn hóa
log1p giảm ảnh hưởng giá trị lớn, không biến dữ liệu thành phân phối chuẩn. K-means tối ưu khoảng cách trên không gian đã biến đổi. Bài này mô tả toàn bộ lịch sử nên fit toàn bộ RFM; nếu dự báo tương lai phải chia theo thời gian trước.


In [ ]:
scaler=StandardScaler()
X=scaler.fit_transform(np.log1p(rfm))
models={k:KMeans(n_clusters=k,n_init=10,random_state=SEED).fit(X) for k in [2,3,4,5]}
comparison=pd.DataFrame([{'k':k,'silhouette':silhouette_score(X,m.labels_,sample_size=min(1000,len(X)),random_state=SEED),'inertia':m.inertia_,'smallest_cluster':int(np.bincount(m.labels_).min())} for k,m in models.items()])
display(comparison)
comparison.plot(x='k',y='silhouette',marker='o'); plt.savefig(OUT/'cluster_selection.png'); plt.show()


## 4. TODO: chọn k và giải thích
Điền k trong 2–5 và một lý do có ít nhất 20 ký tự. Xem đồng thời silhouette, kích thước nhóm và khả năng diễn giải. Không xem silhouette cao nhất là “sự thật” về khách hàng.


In [ ]:
chosen_k = None  # TODO: 2, 3, 4 hoặc 5
reason = ''      # TODO: lý do dựa trên bảng khảo sát
if chosen_k is None: raise NotImplementedError('Chọn k và viết lý do')


In [ ]:
assert chosen_k in models and len(reason.strip())>=20
model=models[chosen_k]
assigned=rfm.assign(cluster=model.labels_)
profile=assigned.groupby('cluster').agg(customers=('recency','size'),recency_median=('recency','median'),frequency_median=('frequency','median'),monetary_median=('monetary','median'))
display(profile)
other=KMeans(n_clusters=chosen_k,n_init=10,random_state=7).fit(X)
print('ARI giữa hai seed:',adjusted_rand_score(model.labels_,other.labels_))
assert len(assigned)==len(rfm)


## 5. Sản phẩm: hồ sơ nhóm và một giả thuyết
Đặt tên nhóm sau khi đọc median trên đơn vị gốc. Nhãn cluster 0/1/2 không có thứ bậc cố hữu. Viết một đề xuất tìm hiểu/kiểm nghiệm cho mỗi nhóm; không tuyên bố phân cụm chứng minh chiến dịch sẽ hiệu quả.


In [ ]:
assigned.to_csv(OUT/'customer_segments.csv')
profile.to_csv(OUT/'segment_profiles.csv')
joblib.dump({'model':model,'scaler':scaler,'columns':list(rfm.columns),'snapshot':'2011-12-10'},OUT/'segments.joblib')
(OUT/'model_card.json').write_text(json.dumps({'k':chosen_k,'reason':reason,'scope':'historical descriptive analysis','transformation':'log1p then standardize','seed':SEED},ensure_ascii=False,indent=2),encoding='utf-8')
for c in profile.index:
    rows=assigned[assigned.cluster.eq(c)]
    plt.scatter(rows.recency,rows.monetary,s=5,alpha=.4,label=str(c))
plt.yscale('log');plt.xlabel('Recency (days)');plt.ylabel('Monetary');plt.legend()
plt.savefig(OUT/'segments.png');plt.show()


## Hoàn thiện sản phẩm và tự kiểm
Loại đơn hủy và số lượng/giá không dương nên monetary không phải doanh thu ròng sau hoàn tiền. Mở rộng: thử quy tắc loại trùng và so sánh độ ổn định; số cụm không phải nhãn thật.

Nộp notebook đã chạy, thư mục đầu ra của dự án và `REPORT.md` tự viết: vấn đề → dữ liệu → phương pháp → kết quả → giới hạn. Dẫn nguồn, ghi seed/cách chia và mô tả ít nhất một trường hợp thất bại. Không chỉ chụp hình điểm số.

**Câu hỏi phản tư:** Điều gì trong kết quả là bằng chứng, điều gì mới là giả thuyết? Nếu dữ liệu mới khác dữ liệu học, bạn cần kiểm tra gì?

**Nhiệm vụ mở:** thay đổi một lựa chọn có lý do, xác định tiêu chí trước khi chạy và so sánh trên validation (hoặc phân tích độ ổn định với bài phân cụm). Không liên tục xem test để chọn cách làm.
